In [1]:
# -*- coding: utf-8 -*-
"""
US required return batch runner (SAFE VERSION - PATCHED)
핵심 개선:
1) FMP 호출 낭비 제거: backfill/forward 시 필요한 구간(from~to)만 호출 (전기간 통째 fetch 금지)
2) 중복 계산/중복 저장 제거: last_saved_date 이후만 저장, 계산은 rolling window 확보용으로만 과거 일부 포함
3) price_stock은 ensure_price_series에서 이미 저장하므로 update_one_ticker에서 재저장 금지(대폭 I/O 감소)
4) DB MIN/MAX 조회는 1회 쿼리로 통합
"""

import os
import time
import math
import requests
import numpy as np
import pandas as pd
import pymysql
import FinanceDataReader as fdr
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple
from pandas.tseries.offsets import BDay
from DATA.us_target_ticker_list_2000 import ticker_list

# =========================================================
# 0) 설정
# =========================================================
DB_NAME = "investar"
TABLE_RESULT = "us_required_return_result"
DEFAULT_PORT = 3307

MARKET_TICKER = "SPY"

MAX_RETRY = 5
SLEEP_BETWEEN_CALLS = 0.35

BETA_WINDOWS = [252, 750, 1250]  # beta_252, beta_750, beta_1250
MAX_ROLLING_WINDOW = max(BETA_WINDOWS)  # 1250
ROLLING_WARMUP_BDAYS = MAX_ROLLING_WINDOW + 80  # rolling 안정화를 위한 여유

STORE_MODE = "minimal"  # "minimal" 또는 "full"

CHECKPOINT_DIR = "_batch_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DEFAULT_DONE_PATH = os.path.join(CHECKPOINT_DIR, "done_tickers.txt")
DEFAULT_FAIL_PATH = os.path.join(CHECKPOINT_DIR, "failed_tickers.txt")


# =========================================================
# 1) DB 연결 / 조회
# =========================================================
def get_conn(db_info: Dict[str, Any]):
    return pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", DEFAULT_PORT),
        user=db_info["user"],
        password=db_info["password"],
        db=db_info.get("database", DB_NAME),
        charset="utf8mb4",
        autocommit=False,
        cursorclass=pymysql.cursors.DictCursor
    )

def ensure_table_pk_hint():
    """
    테이블이 아래와 같은 PK 또는 UNIQUE INDEX를 반드시 가져야 합니다.
    PRIMARY KEY (date, ticker, indicator)
    """
    pass

def get_minmax_date_in_db(
    db_info: Dict[str, Any],
    ticker: str,
    indicator: str
) -> Tuple[Optional[pd.Timestamp], Optional[pd.Timestamp]]:
    """
    MIN/MAX를 한 번에 조회 (커넥션/쿼리 수 감소)
    """
    sql = f"""
    SELECT MIN(date) AS first_date, MAX(date) AS last_date
    FROM {TABLE_RESULT}
    WHERE ticker=%s AND indicator=%s;
    """
    conn = get_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (ticker, indicator))
            row = cur.fetchone()
            first_dt = row["first_date"] if row else None
            last_dt  = row["last_date"] if row else None
    finally:
        conn.close()

    first_dt = pd.to_datetime(first_dt) if first_dt is not None else None
    last_dt  = pd.to_datetime(last_dt)  if last_dt is not None else None
    return first_dt, last_dt

def read_indicator_series(
    db_info: Dict[str, Any],
    ticker: str,
    indicator: str,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
) -> pd.DataFrame:
    """
    returns: columns [date, value]
    필요 구간만 읽을 수 있도록 start/end 옵션 추가(성능 개선)
    """
    where = ["ticker=%s", "indicator=%s"]
    params = [ticker, indicator]

    if start_date is not None:
        where.append("date >= %s")
        params.append(start_date)
    if end_date is not None:
        where.append("date <= %s")
        params.append(end_date)

    sql = f"""
    SELECT date, value
    FROM {TABLE_RESULT}
    WHERE {" AND ".join(where)}
    ORDER BY date;
    """

    conn = get_conn(db_info)
    try:
        df = pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

    if df.empty:
        return pd.DataFrame(columns=["date", "value"])

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.drop_duplicates(subset=["date"]).sort_values("date")
    return df


# =========================================================
# 2) Ticker universe (FDR listing) + 필터
# =========================================================
def get_filtered_us_tickers() -> List[str]:
    nasdaq = fdr.StockListing('NASDAQ')
    nyse = fdr.StockListing('NYSE')
    amex = fdr.StockListing('AMEX')
    info_df = pd.concat([nasdaq, nyse, amex], ignore_index=True)

    exclude_prefixes = ["5510", "5730", "5530", "6010", "5910", "5120"]

    if "IndustryCode" not in info_df.columns or "Symbol" not in info_df.columns:
        tickers = info_df.iloc[:, 0].dropna().astype(str).unique().tolist()
        tickers = [t.strip() for t in tickers if t.strip()]
        return tickers

    info_df["IndustryCode"] = info_df["IndustryCode"].astype(str)
    info_df["IndustryPrefix"] = info_df["IndustryCode"].str[:4]

    mask_exclude = info_df["IndustryPrefix"].isin(exclude_prefixes)
    excluded_df = info_df[mask_exclude]
    filtered_df = info_df[~mask_exclude].copy()

    tickers = filtered_df["Symbol"].dropna().astype(str).unique().tolist()
    tickers = [t.strip() for t in tickers if t.strip()]

    print(f"[INFO] 제외된 기업 수: {len(excluded_df)}")
    print(f"[INFO] 남은 기업 수: {len(filtered_df)}")
    print(f"[INFO] 티커 수: {len(tickers)}")
    return tickers


# =========================================================
# 3) FMP 호출 (가격) - from/to 구간 요청 지원 (낭비 제거)
# =========================================================
def _get_json(url: str, params: Dict[str, Any]) -> Any:
    last_err = None
    for k in range(MAX_RETRY):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 429:
                time.sleep(1.0 + 0.7 * k)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(0.7 + 0.7 * k)
    raise RuntimeError(f"FMP request failed after retries. last_err={last_err}")

def fetch_fdr_price(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    FinanceDataReader로 가격 fallback
    return columns [date, price]
    """
    try:
        df = fdr.DataReader(symbol, start_date, end_date)
    except Exception:
        return pd.DataFrame(columns=["date", "price"])

    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "price"])

    df = df.copy()
    df = df.reset_index().rename(columns={"Date": "date"})
    if "date" not in df.columns:
        # 인덱스가 date였던 케이스
        df.rename(columns={df.columns[0]: "date"}, inplace=True)

    # FDR 컬럼은 보통 Close 또는 Adj Close 등
    if "Adj Close" in df.columns:
        price_col = "Adj Close"
    elif "Close" in df.columns:
        price_col = "Close"
    elif "close" in df.columns:
        price_col = "close"
    else:
        # 마지막 수단: 첫 번째 숫자열
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if not num_cols:
            return pd.DataFrame(columns=["date", "price"])
        price_col = num_cols[0]

    out = df[["date", price_col]].rename(columns={price_col: "price"})
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["price"] = pd.to_numeric(out["price"], errors="coerce")
    out = out.dropna(subset=["date", "price"]).drop_duplicates("date").sort_values("date")
    return out


def fetch_fmp_price(
    symbol: str,
    api_key: str,
    start_date: str,
    end_date: Optional[str] = None,
    min_start_date: str = "2015-01-01"
) -> pd.DataFrame:
    """
    return columns [date, price] (adjClose 우선)
    - start_date가 min_start_date보다 최근이면 min_start_date로 강제 당김
    - end_date(to) 지원: backfill/forward에서 필요한 구간만 받기 위해 필수
    """
    if pd.to_datetime(start_date) > pd.to_datetime(min_start_date):
        start_date = min_start_date

    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
    params = {"from": start_date, "apikey": api_key}
    if end_date is not None:
        params["to"] = end_date

    js = _get_json(url, params=params)
    hist = js.get("historical", []) if isinstance(js, dict) else []
    if not hist:
        return pd.DataFrame(columns=["date", "price"])

    df = pd.DataFrame(hist)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    price_col = "adjClose" if "adjClose" in df.columns else "close"
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
    df = df[["date", price_col]].rename(columns={price_col: "price"})
    df = df.dropna(subset=["price"]).drop_duplicates("date").sort_values("date")
    return df


# =========================================================
# 4) RF(국채 금리) - 배치 1회 다운로드 후 재사용
# =========================================================
def fetch_us_treasury_yields(start_date: str, end_date: Optional[str] = None) -> pd.DataFrame:
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    try:
        y1 = fdr.DataReader("FRED:DGS1", start_date, end_date)
        y3 = fdr.DataReader("FRED:DGS3", start_date, end_date)
        y5 = fdr.DataReader("FRED:DGS5", start_date, end_date)

        idx = y1.index.union(y3.index).union(y5.index)
        out = pd.DataFrame(index=idx).sort_index()
        out["rf_1y"] = pd.to_numeric(y1.iloc[:, 0], errors="coerce") / 100.0
        out["rf_3y"] = pd.to_numeric(y3.iloc[:, 0], errors="coerce") / 100.0
        out["rf_5y"] = pd.to_numeric(y5.iloc[:, 0], errors="coerce") / 100.0
        return out
    except Exception:
        pass

    y1 = fdr.DataReader("^IRX", start_date, end_date)
    y5 = fdr.DataReader("^FVX", start_date, end_date)

    idx = y1.index.union(y5.index)
    out = pd.DataFrame(index=idx).sort_index()
    out["rf_1y"] = pd.to_numeric(y1["Close"], errors="coerce") / 100.0
    out["rf_5y"] = pd.to_numeric(y5["Close"], errors="coerce") / 100.0
    out["rf_3y"] = out["rf_1y"] + (out["rf_5y"] - out["rf_1y"]) * (3 - 1) / (5 - 1)
    return out


# =========================================================
# 5) 계산 (old_version 유지)
# =========================================================
def rolling_beta(ret_stock: pd.Series, ret_mkt: pd.Series, window: int) -> pd.Series:
    cov = ret_stock.rolling(window).cov(ret_mkt)
    var = ret_mkt.rolling(window).var()
    return cov / var

def build_features(price_stock_df: pd.DataFrame, price_mkt_df: pd.DataFrame, rf_df: pd.DataFrame) -> pd.DataFrame:
    df = pd.merge(price_stock_df, price_mkt_df, on="date", how="inner")
    df = df.sort_values("date").drop_duplicates("date")

    df["price_stock"] = pd.to_numeric(df["price_stock"], errors="coerce")
    df["price_mkt"]   = pd.to_numeric(df["price_mkt"], errors="coerce")
    df = df.dropna(subset=["price_stock", "price_mkt"])

    df["ret_stock"] = df["price_stock"].pct_change()
    df["ret_mkt"]   = df["price_mkt"].pct_change()

    for w in BETA_WINDOWS:
        df[f"beta_{w}"] = rolling_beta(df["ret_stock"], df["ret_mkt"], w)

    if rf_df is None or rf_df.empty:
        df["rf_1y"] = np.nan
        df["rf_3y"] = np.nan
        df["rf_5y"] = np.nan
    else:
        rf2 = rf_df.copy().sort_index()
        rf2 = rf2.reindex(pd.to_datetime(df["date"])).ffill()
        df["rf_1y"] = rf2["rf_1y"].values
        df["rf_3y"] = rf2["rf_3y"].values
        df["rf_5y"] = rf2["rf_5y"].values

    df["E_Rm_1y"] = df["ret_mkt"].rolling(252).mean() * 252
    df["E_Rm_3y"] = df["ret_mkt"].rolling(750).mean() * 252
    df["E_Rm_5y"] = df["ret_mkt"].rolling(1250).mean() * 252

    df["Re_1y"] = df["rf_1y"] + df["beta_252"]   * (df["E_Rm_1y"] - df["rf_1y"])
    df["Re_3y"] = df["rf_3y"] + df["beta_750"]   * (df["E_Rm_3y"] - df["rf_3y"])
    df["Re_5y"] = df["rf_5y"] + df["beta_1250"]  * (df["E_Rm_5y"] - df["rf_5y"])

    return df


# =========================================================
# 6) MySQL 저장: NaN/inf 금지 + updated_at 없음 + PK 중복 방지
# =========================================================
def _to_mysql_float(x: Any) -> Optional[float]:
    if x is None:
        return None
    try:
        v = float(x)
    except Exception:
        return None
    if math.isnan(v) or math.isinf(v):
        return None
    return v

def sanitize_long_for_mysql(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out[out["date"].notna()]
    out["date"] = out["date"].dt.date

    out["ticker"] = out["ticker"].astype(str)
    out["indicator"] = out["indicator"].astype(str)

    out = out[(out["ticker"].str.lower() != "nan") & (out["ticker"].str.lower() != "none")]
    out = out[(out["indicator"].str.lower() != "nan") & (out["indicator"].str.lower() != "none")]

    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out["value"] = out["value"].apply(_to_mysql_float)

    out = out.drop_duplicates(subset=["date", "ticker", "indicator"])
    return out

def upsert_long_df(
    db_info: Dict[str, Any],
    long_df: pd.DataFrame,
    batch_size_rows: int = 50_000,
    batch_size_ticker: int = 50,
    drop_null_values: bool = True
) -> None:
    if long_df is None or long_df.empty:
        return

    df = sanitize_long_for_mysql(long_df)
    if drop_null_values:
        df = df.dropna(subset=["value"])
    if df.empty:
        return

    tickers = sorted(df["ticker"].unique())
    n = len(tickers)
    print(f"[INFO] upsert 대상 ticker={n}, rows={len(df):,}")

    insert_sql = f"""
    INSERT INTO {TABLE_RESULT} (date, ticker, indicator, value)
    VALUES (%s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        value = VALUES(value);
    """

    for i in range(0, n, batch_size_ticker):
        batch_tickers = tickers[i:i+batch_size_ticker]
        batch = df[df["ticker"].isin(batch_tickers)].copy()
        batch = batch.sort_values(["ticker", "date", "indicator"])
        rows = list(batch[["date", "ticker", "indicator", "value"]].itertuples(index=False, name=None))

        conn = get_conn(db_info)
        try:
            with conn.cursor() as cur:
                for j in range(0, len(rows), batch_size_rows):
                    chunk = rows[j:j+batch_size_rows]
                    for _r in chunk[:10]:
                        vv = _r[3]
                        if isinstance(vv, float) and (math.isnan(vv) or math.isinf(vv)):
                            raise ValueError(f"Still has NaN/inf in chunk sample: {_r}")
                    cur.executemany(insert_sql, chunk)
                    conn.commit()
            print(f"[OK] batch {i//batch_size_ticker+1}: tickers={len(batch_tickers)}, rows={len(rows):,}")
        except Exception:
            conn.rollback()
            raise
        finally:
            conn.close()


# =========================================================
# 7) 가격 확보: DB price_stock → 부족분만 FMP → DB 저장
#    개선: backfill/forward 모두 필요한 구간(from~to)만 호출
# =========================================================
def ensure_price_series(
    db_info: Dict[str, Any],
    ticker: str,
    api_key: str,
    indicator_name: str = "price_stock",
    today_iso: Optional[str] = None,
    min_start_date: str = "2015-01-01",
    return_start_date: Optional[str] = None,
) -> pd.DataFrame:
    if today_iso is None:
        today_iso = datetime.utcnow().date().isoformat()

    first_dt, last_dt = get_minmax_date_in_db(db_info, ticker, indicator_name)

    # 내부 helper: df_price([date, price]) -> 저장
    def _store_price_df(df_price: pd.DataFrame):
        if df_price is None or df_price.empty:
            return
        df2 = df_price.rename(columns={"price": indicator_name}).copy()
        long_new = df2.assign(ticker=ticker).melt(
            id_vars=["date", "ticker"], value_vars=[indicator_name],
            var_name="indicator", value_name="value"
        )
        upsert_long_df(db_info, long_new, drop_null_values=True)

    # (A) DB에 아무것도 없으면
    if last_dt is None:
        # 1) FMP 시도
        df_new = fetch_fmp_price(
            ticker, api_key,
            start_date=min_start_date,
            end_date=today_iso,
            min_start_date=min_start_date
        )
        time.sleep(SLEEP_BETWEEN_CALLS)

        # 2) FMP가 비면 FDR fallback
        if df_new.empty:
            df_new = fetch_fdr_price(ticker, min_start_date, today_iso)

        if not df_new.empty:
            _store_price_df(df_new)

        out = df_new.rename(columns={"price": indicator_name}).drop_duplicates("date").sort_values("date")
        if return_start_date is not None:
            out = out[out["date"] >= pd.to_datetime(return_start_date)]
        return out

    # (B) 과거 backfill
    if first_dt is not None and first_dt.date().isoformat() > min_start_date:
        back_end = (first_dt - pd.Timedelta(days=1)).date().isoformat()

        df_back = fetch_fmp_price(
            ticker, api_key,
            start_date=min_start_date,
            end_date=back_end,
            min_start_date=min_start_date
        )
        time.sleep(SLEEP_BETWEEN_CALLS)

        if df_back.empty:
            df_back = fetch_fdr_price(ticker, min_start_date, back_end)

        if not df_back.empty:
            _store_price_df(df_back)

    # (C) 최신 forward
    start_missing = (last_dt + pd.Timedelta(days=1)).date().isoformat()
    if start_missing <= today_iso:
        df_fwd = fetch_fmp_price(
            ticker, api_key,
            start_date=start_missing,
            end_date=today_iso,
            min_start_date=min_start_date
        )
        time.sleep(SLEEP_BETWEEN_CALLS)

        if df_fwd.empty:
            df_fwd = fetch_fdr_price(ticker, start_missing, today_iso)

        if not df_fwd.empty:
            _store_price_df(df_fwd)

    # (D) 필요한 구간만 DB에서 읽어서 반환
    df_db = read_indicator_series(
        db_info, ticker, indicator_name,
        start_date=return_start_date, end_date=today_iso
    ).rename(columns={"value": indicator_name})

    df_db["date"] = pd.to_datetime(df_db["date"], errors="coerce")
    df_db = df_db.dropna(subset=["date"]).drop_duplicates("date").sort_values("date")
    return df_db


# =========================================================
# 8) 티커 1개 업데이트 (SPY/RF는 외부에서 주입받아 재사용)
#    개선: last_saved 이후만 저장 / price_stock은 재저장 안 함
# =========================================================
def _pick_reference_indicator(store_mode: str) -> str:
    # 저장 여부 판단을 위한 기준 지표(가급적 rolling이 큰 지표)
    # minimal/full 모두 Re_5y가 가장 보수적(1250 윈도우 필요)
    return "Re_5y"

def update_one_ticker(
    db_info: Dict[str, Any],
    ticker: str,
    api_key: str,
    spy_price_df: pd.DataFrame,   # [date, price_mkt]
    rf_df: pd.DataFrame,          # index=date, columns rf_1y, rf_3y, rf_5y
    today_iso: Optional[str] = None,
    min_start_date: str = "2015-01-01",
    store_mode: str = "minimal",
) -> Tuple[bool, str]:
    if today_iso is None:
        today_iso = datetime.utcnow().date().isoformat()

    if spy_price_df is None or spy_price_df.empty:
        return (False, "SPY price df empty")

    # 0) 마지막 저장일 확인 (중복 계산/저장 방지)
    ref_ind = _pick_reference_indicator(store_mode)
    _, last_saved = get_minmax_date_in_db(db_info, ticker, ref_ind)

    if last_saved is not None:
        # 계산은 rolling window 확보 위해 과거 일부 포함
        calc_start_dt = (pd.to_datetime(last_saved) - BDay(ROLLING_WARMUP_BDAYS)).date().isoformat()
        # 저장은 last_saved 다음날 이후만
        save_after_dt = pd.to_datetime(last_saved).date()
    else:
        calc_start_dt = min_start_date
        save_after_dt = None

    # 1) 종목 가격 확보(필요분만 FMP) + DB 저장
    #    반환도 calc_start_dt 이후만 받도록 제한(성능 개선)
    px_stock = ensure_price_series(
        db_info=db_info,
        ticker=ticker,
        api_key=api_key,
        indicator_name="price_stock",
        today_iso=today_iso,
        min_start_date=min_start_date,
        return_start_date=calc_start_dt
    )
    if px_stock.empty:
        return (False, f"{ticker}: price_stock empty")

    # 2) SPY 가격도 calc_start_dt 이후 구간만 사용
    spy2 = spy_price_df.copy()
    spy2["date"] = pd.to_datetime(spy2["date"], errors="coerce")
    spy2 = spy2.dropna(subset=["date", "price_mkt"])
    spy2 = spy2[spy2["date"] >= pd.to_datetime(calc_start_dt)]
    if spy2.empty:
        return (False, f"{ticker}: SPY slice empty from {calc_start_dt}")

    price_stock_df = px_stock[["date", "price_stock"]].copy()
    price_mkt_df   = spy2[["date", "price_mkt"]].copy()

    # 3) 계산
    feat = build_features(price_stock_df, price_mkt_df, rf_df)
    if feat.empty:
        return (False, f"{ticker}: feature df empty after merge")

    feat["ticker"] = ticker

    # 4) 저장 컬럼 선택
    if store_mode == "full":
        keep_cols = [
            # price_stock은 ensure에서 이미 저장하므로 제외 (중복 저장 방지)
            "price_mkt","ret_stock","ret_mkt",
            "beta_252","beta_750","beta_1250",
            "rf_1y","rf_3y","rf_5y",
            "E_Rm_1y","E_Rm_3y","E_Rm_5y",
            "Re_1y","Re_3y","Re_5y",
        ]
    else:
        keep_cols = [
            # price_stock 제외(중복 저장 방지)
            "beta_252","beta_750","beta_1250",
            "Re_1y","Re_3y","Re_5y",
        ]

    keep_cols = [c for c in keep_cols if c in feat.columns]

    # 5) last_saved 이후만 저장(중복 upsert/DB I/O 제거)
    if save_after_dt is not None:
        feat = feat[pd.to_datetime(feat["date"]).dt.date > save_after_dt]
        if feat.empty:
            return (True, f"{ticker}: up-to-date (no new dates after {save_after_dt})")

    long_df = feat[["date","ticker"] + keep_cols].melt(
        id_vars=["date","ticker"],
        value_vars=keep_cols,
        var_name="indicator",
        value_name="value"
    )

    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["value"])

    if long_df.empty:
        return (False, f"{ticker}: all values NaN after sanitize")

    # 6) 저장
    upsert_long_df(db_info, long_df, drop_null_values=True)
    return (True, f"{ticker}: saved NEW rows={len(long_df):,} ({store_mode}) from {calc_start_dt}")


# =========================================================
# 9) 배치 실행 (체크포인트/실패 재시도 포함)
# =========================================================
def _load_set(path: str) -> set:
    if not os.path.exists(path):
        return set()
    with open(path, "r", encoding="utf-8") as f:
        return set([line.strip() for line in f if line.strip()])

def _append_line(path: str, line: str):
    with open(path, "a", encoding="utf-8") as f:
        f.write(line.strip() + "\n")

def prepare_spy_and_rf(
    db_info: Dict[str, Any],
    api_key: str,
    today_iso: str,
    min_start_date: str = "2015-01-01",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    print("[STEP] Ensure SPY price series (once)")

    spy_px = ensure_price_series(
        db_info=db_info,
        ticker=MARKET_TICKER,
        api_key=api_key,
        indicator_name="price_stock",
        today_iso=today_iso,
        min_start_date=min_start_date,
        return_start_date=min_start_date
    )

    # 여기서도 최종 안전망
    if spy_px.empty:
        print("[WARN] SPY still empty after ensure_price_series. Trying direct FDR fallback...")
        df_fdr = fetch_fdr_price(MARKET_TICKER, min_start_date, today_iso)
        if not df_fdr.empty:
            df2 = df_fdr.rename(columns={"price": "price_stock"})
            long_new = df2.assign(ticker=MARKET_TICKER).melt(
                id_vars=["date", "ticker"], value_vars=["price_stock"],
                var_name="indicator", value_name="value"
            )
            upsert_long_df(db_info, long_new, drop_null_values=True)

            spy_px = df2[["date", "price_stock"]].copy()

    if spy_px.empty:
        raise RuntimeError("SPY price series is empty even after fallback; check API_KEY / network / FDR availability.")

    spy_price_df = spy_px.rename(columns={"price_stock": "price_mkt"})[["date", "price_mkt"]].copy()
    spy_price_df["date"] = pd.to_datetime(spy_price_df["date"])
    spy_price_df["price_mkt"] = pd.to_numeric(spy_price_df["price_mkt"], errors="coerce")
    spy_price_df = spy_price_df.dropna(subset=["date", "price_mkt"]).drop_duplicates("date").sort_values("date")

    start_for_rf = spy_price_df["date"].min().date().isoformat()
    end_for_rf   = spy_price_df["date"].max().date().isoformat()

    print(f"[STEP] Fetch RF once for range: {start_for_rf} ~ {end_for_rf}")
    rf_df = fetch_us_treasury_yields(start_for_rf, end_for_rf)
    if rf_df is None or rf_df.empty:
        print("[WARN] RF df empty; required return may be NaN for many rows.")

    return spy_price_df, rf_df


def run_batch(
    db_info: Dict[str, Any],
    tickers: List[str],
    api_key: str,
    start_idx: int = 0,
    min_start_date: str = "2015-01-01",
    store_mode: str = "minimal",
    done_path: str = DEFAULT_DONE_PATH,
    fail_path: str = DEFAULT_FAIL_PATH,
    skip_done: bool = True,
    retry_failed_once: bool = True,
) -> None:
    today_iso = datetime.utcnow().date().isoformat()

    done_set = _load_set(done_path) if skip_done else set()
    fail_set = set()

    spy_price_df, rf_df = prepare_spy_and_rf(
        db_info=db_info,
        api_key=api_key,
        today_iso=today_iso,
        min_start_date=min_start_date
    )

    total = len(tickers)
    print(f"[RUN] total tickers={total}, start_idx={start_idx}, store_mode={store_mode}, skip_done={skip_done}")

    for idx, t in enumerate(tickers[start_idx:], start=start_idx):
        t = str(t).strip()
        if not t:
            continue
        if skip_done and (t in done_set):
            if (idx % 200) == 0:
                print(f"[SKIP] idx={idx} {t} (already done)")
            continue

        try:
            ok, msg = update_one_ticker(
                db_info=db_info,
                ticker=t,
                api_key=api_key,
                spy_price_df=spy_price_df,
                rf_df=rf_df,
                today_iso=today_iso,
                min_start_date=min_start_date,
                store_mode=store_mode
            )
            if ok:
                print(f"[OK] idx={idx}/{total-1} {msg}")
                _append_line(done_path, t)
                done_set.add(t)
            else:
                print(f"[FAIL] idx={idx}/{total-1} {msg}")
                _append_line(fail_path, t)
                fail_set.add(t)

        except Exception as e:
            print(f"[ERROR] idx={idx}/{total-1} {t}: {e}")
            _append_line(fail_path, t)
            fail_set.add(t)

    if retry_failed_once and fail_set:
        print(f"[RETRY] failed tickers={len(fail_set)} (one more pass)")
        still_fail = set()
        for t in sorted(fail_set):
            try:
                ok, msg = update_one_ticker(
                    db_info=db_info,
                    ticker=t,
                    api_key=api_key,
                    spy_price_df=spy_price_df,
                    rf_df=rf_df,
                    today_iso=today_iso,
                    min_start_date=min_start_date,
                    store_mode=store_mode
                )
                if ok:
                    print(f"[RETRY-OK] {msg}")
                    _append_line(done_path, t)
                    done_set.add(t)
                else:
                    print(f"[RETRY-FAIL] {msg}")
                    still_fail.add(t)
            except Exception as e:
                print(f"[RETRY-ERROR] {t}: {e}")
                still_fail.add(t)

        print(f"[DONE] retry finished. still_fail={len(still_fail)}")
    else:
        print("[DONE] batch finished.")

In [2]:
# if __name__ == "__main__":

from DATA.stock_invest_function import get_db_host

db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",   # 실제 비밀번호
    "database": "investar",
}

API_KEY = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'
tickers = ticker_list
# tickers = ["AMAT"]
run_batch(
    db_info=db_info,
    tickers=tickers,
    api_key=API_KEY,
    start_idx=0,
    min_start_date="2015-01-01",
    store_mode=STORE_MODE,
    done_path=DEFAULT_DONE_PATH,
    fail_path=DEFAULT_FAIL_PATH,
    skip_done=False,          # 단일 테스트 때는 False 추천
    retry_failed_once=True,
)

[STEP] Ensure SPY price series (once)
[INFO] upsert 대상 ticker=1, rows=2,793
[OK] batch 1: tickers=1, rows=2,793
[WARN] SPY still empty after ensure_price_series. Trying direct FDR fallback...
[INFO] upsert 대상 ticker=1, rows=2,794
[OK] batch 1: tickers=1, rows=2,794
[STEP] Fetch RF once for range: 2014-12-31 ~ 2026-02-10
[RUN] total tickers=2000, start_idx=0, store_mode=minimal, skip_done=False
[INFO] upsert 대상 ticker=1, rows=2,793
[OK] batch 1: tickers=1, rows=2,793
[FAIL] idx=0/1999 NVDA: price_stock empty
[INFO] upsert 대상 ticker=1, rows=2,793
[OK] batch 1: tickers=1, rows=2,793
[FAIL] idx=1/1999 GOOG: price_stock empty
[INFO] upsert 대상 ticker=1, rows=2,793
[OK] batch 1: tickers=1, rows=2,793
[FAIL] idx=2/1999 AAPL: price_stock empty
[INFO] upsert 대상 ticker=1, rows=2,793
[OK] batch 1: tickers=1, rows=2,793
[FAIL] idx=3/1999 MSFT: price_stock empty
[INFO] upsert 대상 ticker=1, rows=2,793
[OK] batch 1: tickers=1, rows=2,793
[FAIL] idx=4/1999 AMZN: price_stock empty
[INFO] upsert 대상 ticker

In [7]:
def get_indicator_list_from_db(db_info: Dict,
                               table_name: str = "us_required_return_result"
                               ) -> List[str]:
    """
    큰 테이블 전체를 읽지 않고,
    DB에서 DISTINCT indicator 이름만 추출.
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    query = f"SELECT DISTINCT indicator FROM {table_name};"

    try:
        df = pd.read_sql(query, conn)
    finally:
        conn.close()

    return sorted(df["indicator"].dropna().tolist())


def fetch_required_return_pivot_from_db(
    db_info: Dict,
    table_name: str = "us_required_return_result",
    indicators: Optional[List[str]] = None,
    start_date: Optional[str] = None,   # "YYYY-MM-DD"
    end_date: Optional[str] = None,     # "YYYY-MM-DD"
    tickers: Optional[List[str]] = None
) -> pd.DataFrame:
    """
    us_required_return_result 테이블에서 직접 pivot 형태로 SELECT.

    - index: date, ticker
    - columns: indicator 이름들
    - values: value (MAX(CASE WHEN ...) 사용)

    큰 테이블 전체를 안 읽고, SQL에서 집계해서 가져오기 때문에 메모리 부담이 훨씬 적다.
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        # 1) indicator 리스트가 안 들어오면, DB에서 DISTINCT로 먼저 가져오기
        if indicators is None:
            indicators = get_indicator_list_from_db(db_info, table_name)

        # 2) indicator별 CASE WHEN 절 만들기
        case_clauses = []
        for ind in indicators:
            # 컬럼 alias에 공백/특수문자가 있으면 쿼리가 깨질 수 있으므로 간단히 감싸줌
            safe_alias = ind.replace("`", "")  # 백틱 제거
            clause = f"MAX(CASE WHEN indicator = %s THEN value END) AS `{safe_alias}`"
            case_clauses.append(clause)

        case_sql = ",\n       ".join(case_clauses)

        # 3) WHERE 절 동적으로 생성
        where_clauses = []
        params = []

        if start_date is not None:
            where_clauses.append("date >= %s")
            params.append(start_date)

        if end_date is not None:
            where_clauses.append("date <= %s")
            params.append(end_date)

        if tickers is not None and len(tickers) > 0:
            # IN (%s, %s, ...)
            tick_placeholders = ", ".join(["%s"] * len(tickers))
            where_clauses.append(f"ticker IN ({tick_placeholders})")
            params.extend(tickers)

        where_sql = ""
        if where_clauses:
            where_sql = "WHERE " + " AND ".join(where_clauses)

        # 4) 전체 쿼리 조립
        #    date, ticker별로 indicator를 열로 펼친 형태
        query = f"""
            SELECT
                date,
                ticker,
                {case_sql}
            FROM {table_name}
            {where_sql}
            GROUP BY date, ticker
            ORDER BY date, ticker;
        """

        # indicator 값들을 CASE WHEN의 %s에 넣어줌
        # CASE WHEN indicator = %s THEN value END  → 각 indicator마다 하나씩
        case_params = indicators[:]  # 각 CASE WHEN 한 번씩
        all_params = case_params + params

        df_pivot = pd.read_sql(query, conn, params=all_params)

        # date를 datetime으로 변환 (필요시)
        df_pivot["date"] = pd.to_datetime(df_pivot["date"])
        return df_pivot

    finally:
        conn.close()

In [8]:
item_name = 'roe'

tickers = ['AMAT']

re_df = fetch_required_return_pivot_from_db(
    db_info=db_info,
    table_name="us_required_return_result",
    indicators=['Re_5y'],        # ← 리스트로!
    start_date="2020-01-01",
    end_date=None,
    tickers=tickers
)

print(re_df.head())

        date ticker     Re_5y
0 2020-01-02   AMAT  0.149656
1 2020-01-03   AMAT  0.151032
2 2020-01-06   AMAT  0.148044
3 2020-01-07   AMAT  0.146589
4 2020-01-08   AMAT  0.146287


In [9]:
re_df.tail(10)

,date,ticker,Re_5y
1505,2025-12-29,AMAT,0.242420
1506,2025-12-30,AMAT,0.236434
1507,2025-12-31,AMAT,0.231380
1508,2026-01-02,AMAT,0.234585
1509,2026-01-05,AMAT,0.237361
1510,2026-01-06,AMAT,0.238623
1511,2026-01-07,AMAT,0.239222
1512,2026-01-08,AMAT,0.241437
1513,2026-01-09,AMAT,0.240837
1514,2026-01-12,AMAT,0.236786
